In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/cleantext/__results__.html
/kaggle/input/cleantext/__notebook__.ipynb
/kaggle/input/cleantext/__output__.json
/kaggle/input/cleantext/custom.css
/kaggle/input/cleantext/cleaned_data/items_clean.parquet
/kaggle/input/cleantext/cleaned_data/users_clean.parquet
/kaggle/input/cleantext/cleaned_data/test_ground_truth.parquet
/kaggle/input/cleantext/cleaned_data/transactions_partitioned/month_key=202410/f0d48d85e9f84d908bef9c3a0cea06a4-0.parquet
/kaggle/input/cleantext/cleaned_data/transactions_partitioned/month_key=202408/f0d48d85e9f84d908bef9c3a0cea06a4-0.parquet
/kaggle/input/cleantext/cleaned_data/transactions_partitioned/month_key=202405/f0d48d85e9f84d908bef9c3a0cea06a4-0.parquet
/kaggle/input/cleantext/cleaned_data/transactions_partitioned/month_key=202401/f0d48d85e9f84d908bef9c3a0cea06a4-0.parquet
/kaggle/input/cleantext/cleaned_data/transactions_partitioned/month_key=202402/f0d48d85e9f84d908bef9c3a0cea06a4-0.parquet
/kaggle/input/cleantext/cleaned_data/transactions_part

In [2]:
# ============================================================================
# NOTEBOOK 3: FEATURE ENGINEERING (UPGRADED FOR REPURCHASE)
# ============================================================================
# Status: FEATURE ENHANCED
# New Features:
#  1. item_repurchase_rate: Tỷ lệ món hàng được mua lại.
#  2. user_repurchase_ratio: Tỷ lệ trung thành của khách hàng.
#  3. time_decay_popularity: Popularity có trọng số thời gian (Mới quan trọng hơn cũ).
# ============================================================================

import polars as pl
import numpy as np
import pickle
import os
import gc
import glob
from datetime import datetime
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings

warnings.filterwarnings('ignore')

print("=" * 80)
print("🚀 NOTEBOOK 3: FEATURE ENGINEERING PIPELINE (ENHANCED)")
print("=" * 80)

# ============================================================================
# 1. CONFIGURATION
# ============================================================================
INPUT_DIR = "/kaggle/input/cleantext/cleaned_data"
TRANS_DIR = f"{INPUT_DIR}/transactions_partitioned"
OUTPUT_PATH = "/kaggle/working/feature_engineering"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Ngày cắt Train/Val (Giữ nguyên logic cũ)
TRAIN_END_DATE = "2024-12-31" 

# Config cho CF (Collaborative Filtering)
TOP_K_SIMILAR = 30        
MIN_COOCCURRENCE = 2      
MIN_ITEM_PURCHASES = 5    

print(f"📋 Config:")
print(f"   - Input: {INPUT_DIR}")
print(f"   - Split Date: {TRAIN_END_DATE}")
print(f"   - Min Purchases for CF: {MIN_ITEM_PURCHASES}")

# ============================================================================
# 2. LOAD DATA
# ============================================================================
print("\n[1/11] LOADING DATA...")

df_items = pl.read_parquet(f"{INPUT_DIR}/items_clean.parquet")
df_users = pl.read_parquet(f"{INPUT_DIR}/users_clean.parquet")

print(f"   ✓ Items: {len(df_items):,}")
print(f"   ✓ Users: {len(df_users):,}")

# Load Transactions
trans_files = glob.glob(f"{TRANS_DIR}/**/*.parquet", recursive=True)
if not trans_files:
    trans_files = glob.glob(f"{INPUT_DIR}/transactions*.parquet")

lf_trans = pl.scan_parquet(trans_files)
lf_trans = (
    lf_trans
    .with_columns([
        pl.col("customer_id").cast(pl.Int32),
        pl.col("item_id").cast(pl.Utf8).alias("product_id"),
        pl.col("price").cast(pl.Float64),
        pl.col("quantity").cast(pl.Int32),
        pl.col("timestamp").cast(pl.Int64)
    ])
    .with_columns([
        pl.from_epoch("timestamp", time_unit="s").cast(pl.Date).alias("t_dat"),
        (pl.col("price") * pl.col("quantity")).alias("revenue")
    ])
)
print(f"   ✓ Transactions Linked.")

# ============================================================================
# 3. SPLIT TRAIN / VAL
# ============================================================================
print("\n[2/11] SPLITTING TRAIN/VAL...")
split_date = datetime.strptime(TRAIN_END_DATE, "%Y-%m-%d").date()

lf_train = lf_trans.filter(pl.col("t_dat") <= split_date)
lf_val = lf_trans.filter(pl.col("t_dat") > split_date)

print("   -> Materializing Train Set...")
df_train = lf_train.collect()
print(f"   ✓ Train Data: {len(df_train):,} rows")

print("   -> Saving Validation Set...")
lf_val.collect().write_parquet(f"{OUTPUT_PATH}/validation_data.parquet")

# ============================================================================
# 4. INTERACTION FEATURES (Moved up for Feature Calculation)
# ============================================================================
print("\n[3/11] BUILDING INTERACTION FEATURES (CORE)...")

df_interactions = (
    df_train
    .group_by(["customer_id", "product_id"])
    .agg([
        pl.len().alias("purchase_count"), 
        pl.sum("quantity").alias("total_qty"),
        pl.max("t_dat").alias("last_buy_date")
    ])
    .with_columns([
        (pl.lit(split_date) - pl.col("last_buy_date")).dt.total_days().alias("days_since_last_buy")
    ])
)
print(f"   ✓ Interactions: {len(df_interactions):,} pairs")

# ============================================================================
# 5. CUSTOMER FEATURES (With Repurchase Ratio)
# ============================================================================
print("\n[4/11] BUILDING CUSTOMER FEATURES...")

# 1. Basic Stats
cust_aggs = (
    df_train
    .group_by("customer_id")
    .agg([
        pl.len().alias("transaction_count"),
        pl.sum("revenue").alias("total_revenue"),
        pl.n_unique("product_id").alias("unique_items_bought"),
        pl.max("t_dat").alias("last_purchase_date"),
        pl.mean("revenue").alias("avg_order_value")
    ])
)

# 2. [NEW FEATURE] User Repurchase Ratio
# (Số lượng giao dịch / Số lượng item unique)
# Nếu = 1: Mua mỗi món 1 lần (Thích khám phá)
# Nếu > 1: Hay mua lại món cũ (Trung thành)
cust_repur = (
    df_interactions
    .group_by("customer_id")
    .agg([
        (pl.col("purchase_count") > 1).sum().alias("re-bought_items_count")
    ])
)

df_cust_features = (
    cust_aggs
    .join(df_users, on="customer_id", how="left")
    .join(cust_repur, on="customer_id", how="left")
    .with_columns([
        (pl.lit(split_date) - pl.col("last_purchase_date")).dt.total_days().alias("recency_days"),
        pl.col("region").fill_null("Unknown"),
        pl.col("gender").fill_null("Unknown"),
        pl.col("membership_tier").fill_null("Standard"),
        # Feature mới
        (pl.col("re-bought_items_count") / pl.col("unique_items_bought")).fill_nan(0).alias("user_repurchase_ratio")
    ])
    .drop(["timestamp", "registration_date", "last_purchase_date"], strict=False)
)
print(f"   ✓ Customer Features: {len(df_cust_features):,} (Added: user_repurchase_ratio)")

# ============================================================================
# 6. ITEM FEATURES (With Repurchase Rate)
# ============================================================================
print("\n[5/11] BUILDING ITEM FEATURES...")

# 1. Trending & Popularity
df_trending = (
    df_train
    .with_columns([(pl.lit(split_date) - pl.col("t_dat")).dt.total_days().alias("days_diff")])
    .with_columns([(1.0 / (pl.col("days_diff") + 1.0)).alias("decay_weight")])
    .group_by("product_id")
    .agg([
        pl.sum("decay_weight").alias("trending_score_30d"),
        pl.len().alias("popularity_score") 
    ])
)

# 2. [NEW FEATURE] Item Repurchase Rate
# (Số lượng user mua lại món này / Tổng số user mua món này)
item_repur_stats = (
    df_interactions
    .group_by("product_id")
    .agg([
        pl.n_unique("customer_id").alias("unique_buyers"),
        (pl.col("purchase_count") > 1).sum().alias("repeat_buyers_count")
    ])
    .with_columns([
        (pl.col("repeat_buyers_count") / pl.col("unique_buyers")).fill_nan(0).alias("item_repurchase_rate")
    ])
)

# 3. Basic Aggs
item_aggs = df_train.group_by("product_id").agg([
    pl.sum("revenue").alias("total_sales")
])

# 4. Merge All
df_item_features = (
    item_aggs
    .join(df_trending, on="product_id", how="left")
    .join(item_repur_stats, on="product_id", how="left") # Join repurchase stats
    .join(df_items.rename({"item_id": "product_id"}), on="product_id", how="left")
    .with_columns([
        (pl.col("total_sales") / pl.col("popularity_score")).alias("avg_real_price"),
        pl.col("trending_score_30d").fill_null(0),
        pl.col("item_repurchase_rate").fill_null(0)
    ])
)

# 5. Price Segment
q33 = df_item_features["avg_real_price"].quantile(0.33)
q66 = df_item_features["avg_real_price"].quantile(0.66)
df_item_features = df_item_features.with_columns(
    pl.when(pl.col("avg_real_price") <= q33).then(pl.lit("Low"))
    .when(pl.col("avg_real_price") <= q66).then(pl.lit("Mid"))
    .otherwise(pl.lit("High")).alias("price_segment")
)
print(f"   ✓ Item Features Enhanced (Added: item_repurchase_rate)")

# ============================================================================
# 7. EXPORT USER HISTORY (REPURCHASE DICT)
# ============================================================================
print("\n[6/11] EXPORTING USER HISTORY...")

history_df = (
    df_interactions
    .sort(["customer_id", "purchase_count", "last_buy_date"], descending=[False, True, True])
    .group_by("customer_id")
    .agg(pl.col("product_id"))
)

user_history_map = {}
for row in history_df.iter_rows(named=True):
    user_history_map[row['customer_id']] = row['product_id']

with open(f"{OUTPUT_PATH}/user_purchase_history.pkl", "wb") as f:
    pickle.dump(user_history_map, f)
print(f"   ✓ Saved user_purchase_history.pkl")

# ============================================================================
# 8. CO-OCCURRENCE MATRIX
# ============================================================================
print("\n[7/11] GENERATING CO-OCCURRENCE...")

top_items_list = df_item_features.sort("popularity_score", descending=True).head(5000)["product_id"].to_list()
basket_df = df_train.filter(pl.col("product_id").is_in(top_items_list)).select(["customer_id", "t_dat", "product_id"]).unique()

cooc_df = (
    basket_df.join(basket_df, on=["customer_id", "t_dat"], suffix="_right")
    .filter(pl.col("product_id") != pl.col("product_id_right"))
    .group_by(["product_id", "product_id_right"])
    .agg(pl.len().alias("co_count"))
    .filter(pl.col("co_count") >= MIN_COOCCURRENCE)
)

cooc_dict = {}
for row in cooc_df.iter_rows(named=True):
    p1, p2, c = row['product_id'], row['product_id_right'], row['co_count']
    if p1 not in cooc_dict: cooc_dict[p1] = []
    cooc_dict[p1].append((p2, c))

for k in cooc_dict:
    cooc_dict[k] = sorted(cooc_dict[k], key=lambda x: x[1], reverse=True)[:TOP_K_SIMILAR]

with open(f"{OUTPUT_PATH}/cooccurrence_topk.pkl", "wb") as f:
    pickle.dump(cooc_dict, f)
print(f"   ✓ Saved Co-occurrence.")
del basket_df, cooc_df
gc.collect()

# ============================================================================
# 9. ITEM SIMILARITY (COSINE)
# ============================================================================
print("\n[8/11] GENERATING COSINE SIMILARITY...")

valid_items = df_item_features.filter(pl.col("popularity_score") >= MIN_ITEM_PURCHASES)["product_id"].to_list()
df_cf = df_train.filter(pl.col("product_id").is_in(valid_items))

users = df_cf["customer_id"].unique().sort().to_list()
items = valid_items
u_map = {u: i for i, u in enumerate(users)}
i_map = {item: i for i, item in enumerate(items)}

rows = [u_map[u] for u in df_cf["customer_id"]]
cols = [i_map[i] for i in df_cf["product_id"]]
data = [1] * len(rows)
mat = csr_matrix((data, (rows, cols)), shape=(len(users), len(items)))

print("   -> Computing Matrix...")
sim_mat = cosine_similarity(mat.T, dense_output=False)

cosine_dict = {}
for idx in range(len(items)):
    row = sim_mat[idx]
    if row.nnz == 0: continue
    indices, scores = row.indices, row.data
    sorted_idx = np.argsort(scores)[::-1]
    
    top = []
    cnt = 0
    for i in sorted_idx:
        tgt = indices[i]
        if tgt == idx: continue
        top.append((items[tgt], float(scores[i])))
        cnt += 1
        if cnt >= TOP_K_SIMILAR: break
    if top: cosine_dict[items[idx]] = top

with open(f"{OUTPUT_PATH}/item_similarity_cosine.pkl", "wb") as f:
    pickle.dump(cosine_dict, f)
print(f"   ✓ Saved Cosine Similarity.")
del mat, sim_mat
gc.collect()

# ============================================================================
# 10. COLD START MAPS
# ============================================================================
print("\n[9/11] GENERATING COLD START MAPS...")

df_pop = df_train.join(df_cust_features.select(["customer_id", "region", "gender", "membership_tier"]), on="customer_id")

def get_tops(df, grp_cols, k=100):
    desc_rules = [False] * len(grp_cols) + [True]
    return (
        df.group_by(grp_cols + ["product_id"])
        .agg(pl.len().alias("count"))
        .sort(grp_cols + ["count"], descending=desc_rules)
        .group_by(grp_cols)
        .agg(pl.col("product_id").head(k))
    )

cold_maps = {}
cold_maps["global"] = df_item_features.sort("trending_score_30d", descending=True).head(100)["product_id"].to_list()
r_df = get_tops(df_pop, ["region"])
cold_maps["region"] = {r["region"]: r["product_id"] for r in r_df.iter_rows(named=True)}
rg_df = get_tops(df_pop, ["region", "gender"])
cold_maps["region_gender"] = {f"{r['region']}_{r['gender']}": r["product_id"] for r in rg_df.iter_rows(named=True)}
mem_df = get_tops(df_pop, ["membership_tier"])
cold_maps["membership"] = {r["membership_tier"]: r["product_id"] for r in mem_df.iter_rows(named=True)}

with open(f"{OUTPUT_PATH}/cold_start_maps.pkl", "wb") as f:
    pickle.dump(cold_maps, f)
print(f"   ✓ Saved Cold Start Maps.")

# ============================================================================
# 11. CONTENT SIMILARITY
# ============================================================================
print("\n[10/11] CONTENT SIMILARITY...")

df_content = df_item_features.select(["product_id", "category_l1", "category_l2", "brand"]).with_columns(
    pl.concat_str([pl.col("category_l1").fill_null(""), pl.col("category_l2").fill_null(""), pl.col("brand").fill_null("")], separator=" ").alias("text")
)
ids = df_content["product_id"].to_list()
tfidf = TfidfVectorizer(max_features=1000, stop_words="english")
tf_mat = tfidf.fit_transform(df_content["text"].to_list())
cont_sim = cosine_similarity(tf_mat, dense_output=False)

cont_dict = {}
for idx in range(len(ids)):
    sc = cont_sim[idx].data
    ind = cont_sim[idx].indices
    s_idx = np.argsort(sc)[::-1]
    top = []
    cnt = 0
    for i in s_idx:
        real = ind[i]
        if real == idx: continue
        top.append((ids[real], float(sc[i])))
        cnt += 1
        if cnt >= TOP_K_SIMILAR: break
    cont_dict[ids[idx]] = top

with open(f"{OUTPUT_PATH}/content_similarity.pkl", "wb") as f:
    pickle.dump(cont_dict, f)

# ============================================================================
# 12. FINALIZE
# ============================================================================
print("\n[11/11] SAVING FINAL PARQUETS...")
df_cust_features.write_parquet(f"{OUTPUT_PATH}/customer_features.parquet")
df_item_features.write_parquet(f"{OUTPUT_PATH}/item_features.parquet")
df_interactions.write_parquet(f"{OUTPUT_PATH}/interaction_features.parquet")

gt_path = f"{INPUT_DIR}/test_ground_truth.parquet"
if os.path.exists(gt_path):
    pl.read_parquet(gt_path).write_parquet(f"{OUTPUT_PATH}/test_ground_truth.parquet")

print("\n" + "="*80)
print("✅ COMPLETED SUCCESSFULLY!")
print(f"Output: {OUTPUT_PATH}")
print("New Features Added:")
print("  - item_repurchase_rate (In item_features)")
print("  - user_repurchase_ratio (In customer_features)")
print("="*80)

🚀 NOTEBOOK 3: FEATURE ENGINEERING PIPELINE (ENHANCED)
📋 Config:
   - Input: /kaggle/input/cleantext/cleaned_data
   - Split Date: 2024-12-31
   - Min Purchases for CF: 5

[1/11] LOADING DATA...
   ✓ Items: 15,972
   ✓ Users: 1,174,236
   ✓ Transactions Linked.

[2/11] SPLITTING TRAIN/VAL...
   -> Materializing Train Set...
   ✓ Train Data: 33,113,674 rows
   -> Saving Validation Set...

[3/11] BUILDING INTERACTION FEATURES (CORE)...
   ✓ Interactions: 22,305,298 pairs

[4/11] BUILDING CUSTOMER FEATURES...
   ✓ Customer Features: 1,153,598 (Added: user_repurchase_ratio)

[5/11] BUILDING ITEM FEATURES...
   ✓ Item Features Enhanced (Added: item_repurchase_rate)

[6/11] EXPORTING USER HISTORY...
   ✓ Saved user_purchase_history.pkl

[7/11] GENERATING CO-OCCURRENCE...
   ✓ Saved Co-occurrence.

[8/11] GENERATING COSINE SIMILARITY...
   -> Computing Matrix...
   ✓ Saved Cosine Similarity.

[9/11] GENERATING COLD START MAPS...
   ✓ Saved Cold Start Maps.

[10/11] CONTENT SIMILARITY...

[11/1

In [3]:
# Chèn đoạn này vào để check (Debug)
dist = df_item_features.select("popularity_score").to_pandas()

n_total = len(dist)
n_less_5 = len(dist[dist['popularity_score'] < 5])
n_less_3 = len(dist[dist['popularity_score'] < 3])

print(f"Tổng số Items: {n_total}")
print(f"Items < 5 lượt mua: {n_less_5} ({n_less_5/n_total:.1%}) -> Sẽ bị loại khỏi CF, dùng Content-Based bù vào.")
print(f"Items < 3 lượt mua: {n_less_3} ({n_less_3/n_total:.1%})")

Tổng số Items: 15782
Items < 5 lượt mua: 150 (1.0%) -> Sẽ bị loại khỏi CF, dùng Content-Based bù vào.
Items < 3 lượt mua: 56 (0.4%)
